# Transformer input ablations: Ddc / full Yb / imaginary Yb / real Yb

Updated interface for `dl_normalized_datasets` v2. Set CASE=30, 39, or 57. The terminal runner and notebook use the same code. Run from this code folder.


In [4]:
from pathlib import Path
import subprocess
import sys
import json
import csv

CODE_DIR = Path.cwd()  # Folder containing train_transformer_imlive.py and updated transformer files
DATA_ROOT = Path('/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset').resolve()  # EDIT if needed
CASE = 57              # 30, 39, or 57
TARGET = 'dq'          # 'dq' or 'vmag'; one target type per run
MODELS = ['axial']  # Add 'dq' to run the global-attention model too
VARIANTS = ['ddc', 'ddc_yb', 'ddc_imag', 'ddc_real']
SEEDS = [42, 123, 2026]
EPOCHS = 200
BATCH_SIZE = 16
D_MODEL = 96
DEPTH = 4
NUM_HEADS = 6
DROPOUT = 0.1
WARMUP_EPOCHS = 5
LOSS_ALPHA = 0.5       # 1.0 means pure MSE
DEVICE = 'cuda'
SMOKE = False         # True uses tiny subsets/two epochs; not a performance estimate
RESUME = True
AMP = True

DATA_DIR = DATA_ROOT / f'case{CASE}'
OUT_DIR = CODE_DIR / ('smoke_results' if SMOKE else 'results') / f'case{CASE}_{TARGET}'
assert (CODE_DIR / 'train_transformer_imlive.py').is_file(), 'Set CODE_DIR to the extracted updated code folder'
for name in ['train', 'val', 'test']:
    assert (DATA_DIR / f'dataset_{name}.npz').is_file(), f'Missing {name} data'
print('Data:', DATA_DIR)
print('Results:', OUT_DIR)


Data: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case57
Results: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer/results/case57_dq


In [5]:
command = [sys.executable, '-u', str(CODE_DIR / 'train_transformer_imlive.py'),
    '--data-dir', str(DATA_DIR), '--out-dir', str(OUT_DIR),
    '--models', *MODELS, '--variants', *VARIANTS,
    '--target', TARGET, '--seeds', *map(str, SEEDS),
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
    '--loss-alpha', str(LOSS_ALPHA), '--device', DEVICE,
    '--d-model', str(D_MODEL), '--depth', str(DEPTH), '--num-heads', str(NUM_HEADS),
    '--dropout', str(DROPOUT), '--warmup-epochs', str(WARMUP_EPOCHS)]
if SMOKE: command.append('--smoke')
if RESUME: command.append('--resume')
if not AMP: command.append('--no-amp')
print(' '.join(command))
subprocess.run(command, cwd=CODE_DIR, check=True)


/opt/tljh/user/bin/python -u /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer/train_transformer_imlive.py --data-dir /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case57 --out-dir /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer/results/case57_dq --models axial --variants ddc ddc_yb ddc_imag ddc_real --target dq --seeds 42 123 2026 --epochs 200 --batch-size 16 --loss-alpha 0.5 --device cuda --d-model 96 --depth 4 --num-heads 6 --dropout 0.1 --warmup-epochs 5 --resume
PyTorch 2.10.0+cu128 device cuda
Dataset(dq, X=(6713, 3, 57, 57), y=(6713, 1, 57, 57)) 
 Dataset(dq, X=(1438, 3, 57, 57), y=(1438, 1, 57, 57))

axial ddc seed=42: 906,337 parameters; AMP=True

axial ddc_yb seed=42: 906,337 parameters; AMP=True

axial ddc_imag seed=42: 906,337 parameters; AMP=True

axial ddc_real seed=42: 906,337 parameters; AMP=True

axial ddc seed=123: 906,337 parameters; AMP=True

axial ddc_yb seed=123

CompletedProcess(args=['/opt/tljh/user/bin/python', '-u', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer/train_transformer_imlive.py', '--data-dir', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case57', '--out-dir', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer/results/case57_dq', '--models', 'axial', '--variants', 'ddc', 'ddc_yb', 'ddc_imag', 'ddc_real', '--target', 'dq', '--seeds', '42', '123', '2026', '--epochs', '200', '--batch-size', '16', '--loss-alpha', '0.5', '--device', 'cuda', '--d-model', '96', '--depth', '4', '--num-heads', '6', '--dropout', '0.1', '--warmup-epochs', '5', '--resume'], returncode=0)

In [6]:
with (OUT_DIR / 'summary.json').open() as f:
    summary = json.load(f)
print('Metrics use original units:', summary['protocol']['metric_units'])
for model, variants in summary['aggregate'].items():
    for variant, splits in variants.items():
        print(model, variant, 'test:', splits['test'])
print('Training-mean baseline:', summary['training_mean_baseline'])
print('Individual runs:', OUT_DIR / 'comparison.csv')


Metrics use original units: raw DQ
axial ddc test: {'r2': {'mean': 0.8611966755348823, 'std_across_seeds': 0.0002319347179718348}, 'rmse': {'mean': 2.3680031742630203, 'std_across_seeds': 0.001978371539744111}, 'mae': {'mean': 0.8981457058231812, 'std_across_seeds': 0.011448736399586401}}
axial ddc_yb test: {'r2': {'mean': 0.8674224354739176, 'std_across_seeds': 0.0005469703165407758}, 'rmse': {'mean': 2.3142850448048295, 'std_across_seeds': 0.004776693082750546}, 'mae': {'mean': 0.8877638105303576, 'std_across_seeds': 0.004519391405557491}}
axial ddc_imag test: {'r2': {'mean': 0.8632760245154009, 'std_across_seeds': 0.0001625951083630462}, 'rmse': {'mean': 2.350199529419171, 'std_across_seeds': 0.0013972700459553042}, 'mae': {'mean': 0.9043697177490505, 'std_across_seeds': 0.018363911587807605}}
axial ddc_real test: {'r2': {'mean': 0.8643735658328331, 'std_across_seeds': 0.0006239114255281276}, 'rmse': {'mean': 2.3407436504904133, 'std_across_seeds': 0.005381176428202376}, 'mae': {'me

## Notes

All four variants share the same fitted training normalization, parameter count within each architecture, initial weights per seed, and training protocol. Excluded slots are masked after normalization. Test data is evaluated only after validation checkpoint selection. Do not apply a second scaler or sigmoid to standardized targets. The saved predictions and R2/RMSE/MAE are in original units. Axial attention is the default. Global attention over B*B cells has much higher memory use; reduce batch size if selecting dq. See README.md for inference and recovery.
